## Desigualdades de Gênero na Ciência de Dados (2021–2024)

**Estratégia:**
- **Treino:** anos 2021, 2022 e 2023 (dados sem perguntas de IA)
- **Teste / drift:** 2024 (inclui colunas de adoção de IA generativa)
- **Targets:** Faixa Salarial · Nível de Senioridade · Cargo de Liderança
- **Modelos:** Regressão Logística · Random Forest · XGBoost
- **Explicabilidade:** SHAP

## 1. Imports e configurações

In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (classification_report, accuracy_score, f1_score)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

SEED = 42
np.random.seed(SEED)
os.makedirs('figures', exist_ok=True)

print("✅ Imports OK")

✅ Imports OK


In [9]:
df_full = pd.read_excel("df_full.xlsx")

## 2. Definição de features

In [10]:
# Ordens das variáveis ordinais (para encoding consistente)
FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k', 'R$2k-3k', 'R$3k-4k', 'R$4k-6k', 'R$6k-8k',
    'R$8k-12k', 'R$12k-16k', 'R$16k-20k', 'R$20k-25k',
    'R$25k-30k', 'R$30k-40k', 'R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal', 'Estudante de Graduação',
    'Graduação/Bacharelado', 'Especialização Lato Sensu',
    'Mestrado', 'Doutorado ou Phd',
]
NIVEL_SENIORIDADE_ORDEM = ['Júnior', 'Pleno', 'Sênior', 'Gestor']

# ── Features base (disponíveis em todos os anos) ─────────────────────────────
# nivel_ensino é tratada como numérica (ordinal encoding antes do pipeline)
BASE_FEATURES = [
    'nivel_ensino',      # ordinal → int
    'area_formacao',     # nominal
    'cargo',             # nominal
    'tempo_area_dados',  # nominal (faixas de tempo)
    'tempo_area_ti',     # nominal (faixas de tempo)
    'modalidade',        # nominal
    'setor',             # nominal
    'regiao',            # nominal
    'situacao',          # nominal
]

# ── Features de IA (exclusivas do 2024) ─────────────────────────────────────
#
# Agrupadas por bloco para facilitar análises parciais:
#   IA_FEATS_GEST  → perspectiva do gestor sobre uso de IA na empresa   (3.f)
#   IA_FEATS_BAR   → barreiras para não usar IA, perspectiva do gestor  (3.g)
#   IA_FEATS_IND   → perspectiva individual sobre uso de IA na empresa  (4.l)
#   IA_FEATS_PES   → uso pessoal de ChatGPT/Copilot                     (4.m)
#   IA_FEATS_ALL   → todos os grupos juntos

IA_FEATS_GEST = [f'ia_gest_{s}' for s in
                 ['independente','centralizado','copilot_devs',
                  'prod_externo','prod_interno','frente_negocio',
                  'nao_prioridade','nao_sei']]

IA_FEATS_BAR  = [f'ia_bar_{s}' for s in
                 ['casos_uso','alucinacao','regulacao','privacidade',
                  'roi','dados','expertise','alta_direcao','prop_intelectual']]

IA_FEATS_IND  = [f'ia_ind_{s}' for s in
                 ['independente','centralizado','copilot_devs',
                  'prod_externo','prod_interno','frente_negocio',
                  'nao_prioridade','nao_sei']]

IA_FEATS_PES  = ['ia_pes_nao_usa','ia_pes_gratis','ia_pes_pago_proprio',
                 'ia_pes_pago_empresa','ia_pes_copilot']

IA_FEATS_ALL  = IA_FEATS_GEST + IA_FEATS_BAR + IA_FEATS_IND + IA_FEATS_PES

# Targets
TARGETS = {
    'faixa_salarial': 'multiclass',
    'nivel':          'multiclass',
    'gestor':         'binario',
}

TARGET_LABELS = {
    'faixa_salarial': 'Faixa Salarial',
    'nivel':          'Senioridade',
    'gestor':         'Gestor/a',
}

print(f"Base features:   {len(BASE_FEATURES)}")
print(f"IA features (3.f):  {len(IA_FEATS_GEST)}")
print(f"IA features (3.g):  {len(IA_FEATS_BAR)}")
print(f"IA features (4.l):  {len(IA_FEATS_IND)}")
print(f"IA features (4.m):  {len(IA_FEATS_PES)}")
print(f"IA features total:  {len(IA_FEATS_ALL)}")


Base features:   9
IA features (3.f):  8
IA features (3.g):  9
IA features (4.l):  8
IA features (4.m):  5
IA features total:  30


## 3. Funções Auxiliares


In [11]:
# ── Encoding ─────────────────────────────────────────────────────────────────

def encode_ordinal(series: pd.Series, ordem: list) -> pd.Series:
    """Converte variável ordinal em int. Valores fora da ordem → NaN."""
    cat   = pd.Categorical(series, categories=ordem, ordered=True)
    codes = cat.codes.astype(float)
    codes[codes == -1] = np.nan
    return codes


def prepare_xy(df: pd.DataFrame, target: str,
               extra_features: list = None) -> tuple:
    """
    Prepara X e y para um target.

    - nivel_ensino é codificada como int ordinal antes de entrar no pipeline.
    - Demais categóricas ficam como str/object (OHE no ColumnTransformer).
    - extra_features: colunas adicionais (e.g. features de IA).
    - Retorna X, y e meta (genero, ano_pesquisa) para análise de equidade.
    """
    feats = BASE_FEATURES.copy()
    if extra_features:
        feats += [f for f in extra_features if f in df.columns]

    sub = df[feats + [target, 'ano_pesquisa', 'genero']].copy()

    # Ordinal encoding de nivel_ensino
    sub['nivel_ensino'] = encode_ordinal(sub['nivel_ensino'], NIVEL_ENSINO_ORDEM)

    # Target encoding
    if target == 'faixa_salarial':
        sub['y'] = encode_ordinal(sub['faixa_salarial'], FAIXA_SALARIAL_ORDEM)
    elif target == 'nivel':
        sub['y'] = encode_ordinal(sub['nivel'], NIVEL_SENIORIDADE_ORDEM)
    elif target == 'gestor':
        sub['y'] = (sub['gestor'] == 1.0).astype(float)

    sub = sub.dropna(subset=['y'])
    sub['y'] = sub['y'].astype(int)

    X    = sub[feats].copy()
    y    = sub['y']
    meta = sub[['genero', 'ano_pesquisa']].copy()
    return X, y, meta



In [12]:

# ── Pipeline sklearn ──────────────────────────────────────────────────────────

def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """
    ColumnTransformer automático baseado nos dtypes de X:
    - Numéricas  → impute(mediana) + StandardScaler
    - Categóricas → impute(moda) + OneHotEncoder(handle_unknown='ignore')
    """
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

    transformers = []
    if num_cols:
        transformers.append(('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scl', StandardScaler()),
        ]), num_cols))
    if cat_cols:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols))

    return ColumnTransformer(transformers, remainder='drop')


def get_models(task_type: str) -> dict:
    """Os 3 modelos configurados para classificação binária ou multiclasse."""
    if task_type == 'binario':
        return {
            'Regressão Logística': LogisticRegression(
                max_iter=500, random_state=SEED, class_weight='balanced'),
            'Random Forest': RandomForestClassifier(
                n_estimators=200, random_state=SEED,
                class_weight='balanced', n_jobs=-1),
            'XGBoost': XGBClassifier(
                n_estimators=200, random_state=SEED,
                eval_metric='logloss', scale_pos_weight=5, verbosity=0),
        }
    else:
        return {
            'Regressão Logística': LogisticRegression(
                max_iter=500, random_state=SEED, class_weight='balanced',
                multi_class='multinomial'),
            'Random Forest': RandomForestClassifier(
                n_estimators=200, random_state=SEED,
                class_weight='balanced', n_jobs=-1),
            'XGBoost': XGBClassifier(
                n_estimators=200, random_state=SEED,
                eval_metric='mlogloss', verbosity=0),
        }




In [13]:
# ── Cross-validation ──────────────────────────────────────────────────────────

def run_cv(X: pd.DataFrame, y: pd.Series, models: dict,
           cv: int = 5) -> pd.DataFrame:
    """
    Cross-validation estratificado (F1-macro).
    Retorna DataFrame com mean e std por modelo.
    """
    skf     = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    results = []
    for name, model in models.items():
        pre  = build_preprocessor(X)
        pipe = Pipeline([('pre', pre), ('clf', model)])
        scores = cross_val_score(pipe, X, y, cv=skf,
                                 scoring='f1_macro', n_jobs=-1)
        results.append({
            'Modelo':         name,
            'F1_macro_mean':  scores.mean(),
            'F1_macro_std':   scores.std(),
        })
        print(f"    {name:25s}: {scores.mean():.3f} ± {scores.std():.3f}")
    return pd.DataFrame(results)

In [14]:
# ── Treino + avaliação ────────────────────────────────────────────────────────

def treinar_avaliar(target: str, task_type: str,
                    df_train: pd.DataFrame, df_test: pd.DataFrame,
                    extra_features: list = None,
                    label_extra: str = '') -> dict:
    """
    1. Cross-validation no treino (seleciona melhor modelo por F1-macro)
    2. Retreina no treino completo
    3. Avalia no teste
    4. Quebra desempenho por gênero (equidade / model drift)
    Retorna dict com tudo necessário para plots e SHAP.
    """
    tag = f'{TARGET_LABELS[target]}{" " + label_extra if label_extra else ""}'
    print(f'\n{"+"*60}')
    print(f'  {tag}')
    print(f'{"+"*60}')

    X_train, y_train, _          = prepare_xy(df_train, target, extra_features)
    X_test,  y_test,  meta_test  = prepare_xy(df_test,  target, extra_features)

    print(f'  Treino: {len(X_train):,} | Teste: {len(X_test):,} | Classes: {sorted(y_train.unique())}')

    # CV
    print(f'\n  [Cross-Validation 5-fold]')
    models   = get_models(task_type)
    cv_df    = run_cv(X_train, y_train, models)
    best_name = cv_df.loc[cv_df['F1_macro_mean'].idxmax(), 'Modelo']
    print(f'\n  Melhor modelo: {best_name}')

    # Treina no full train
    pre  = build_preprocessor(X_train)
    pipe = Pipeline([('pre', pre), ('clf', models[best_name])])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # Relatório geral
    print(f'\n  [Classificação — Teste]')
    print(classification_report(y_test, y_pred, zero_division=0))

    # Equidade por gênero
    gender_rows = []
    for g in ['Masculino', 'Feminino']:
        mask = meta_test['genero'] == g
        if mask.sum() == 0:
            continue
        gender_rows.append({
            'Gênero':    g,
            'N':         int(mask.sum()),
            'Acurácia':  accuracy_score(y_test[mask], y_pred[mask]),
            'F1_macro':  f1_score(y_test[mask], y_pred[mask],
                                   average='macro', zero_division=0),
        })

    print(f'\n  [Desempenho por Gênero]')
    print(pd.DataFrame(gender_rows).to_string(index=False))

    return {
        'target':      target,
        'label':       tag,
        'best_name':   best_name,
        'pipe':        pipe,
        'cv_df':       cv_df,
        'X_train':     X_train,
        'X_test':      X_test,
        'y_test':      y_test,
        'y_pred':      y_pred,
        'gender_rows': gender_rows,
        'meta_test':   meta_test,
    }




In [15]:
def encode_ordinal(series, ordem):
    """Codifica variável ordinal como int. Categorias não mapeadas → NaN."""
    cat = pd.Categorical(series, categories=ordem, ordered=True)
    codes = cat.codes.astype(float)
    codes[codes == -1] = np.nan
    return codes


def prepare_xy(df: pd.DataFrame, target: str, extra_features: list = None):
    """
    Prepara X e y para um dado target.
    - nivel_ensino é codificada como int (ordinal).
    - Demais features permanecem como str/object para OHE no pipeline.
    - Retorna também metadados (genero, ano) para análise de equidade.
    """
    feats = BASE_FEATURES.copy()
    if extra_features:
        feats += [f for f in extra_features if f in df.columns]

    sub = df[feats + [target, 'ano_pesquisa', 'genero']].copy()

    # Ordinal encoding de nivel_ensino
    sub['nivel_ensino'] = encode_ordinal(sub['nivel_ensino'], NIVEL_ENSINO_ORDEM)

    # Target
    if target == 'faixa_salarial':
        sub['y'] = encode_ordinal(sub['faixa_salarial'], FAIXA_SALARIAL_ORDEM)
    elif target == 'nivel':
        sub['y'] = encode_ordinal(sub['nivel'], NIVEL_SENIORIDADE_ORDEM)
    elif target == 'gestor':
        sub['y'] = (sub['gestor'] == 1.0).astype(float)

    sub = sub.dropna(subset=['y'])
    sub['y'] = sub['y'].astype(int)

    X = sub[feats].copy()
    y = sub['y']
    meta = sub[['genero', 'ano_pesquisa']]
    return X, y, meta


print("✅ Feature engineering definido")


✅ Feature engineering definido


## 4. Carregamento dos parquets

In [16]:
f_train = pd.read_parquet('data/df_train.parquet')
df_test  = pd.read_parquet('data/df_test.parquet')

print(f"df_train: {len(df_train):,} linhas  ({df_train['ano_pesquisa'].unique()})")
print(f"df_test:  {len(df_test):,}  linhas  ({df_test['ano_pesquisa'].unique()})")

# Verifica features de IA disponíveis no teste
ia_disponivel = [c for c in IA_FEATS_ALL if c in df_test.columns
                 and df_test[c].notna().sum() > 0]
print(f"\nFeatures de IA disponíveis no df_test: {len(ia_disponivel)}/{len(IA_FEATS_ALL)}")


df_train: 0 linhas  ([])
df_test:  5,178  linhas  (['2024'])

Features de IA disponíveis no df_test: 0/30


## 5. Modelo A - Treino 2021-2023 / Teste 2024 

In [17]:
# Objetivo: verificar se os modelos treinados sem IA ainda conseguem predizer bem em 2024,  
# e se o desempenho difere entre homens e mulheres (model drift de equidade)

results_A = {}
for target, task_type in TARGETS.items():
    results_A[target] = treinar_avaliar(
        target, task_type, df_train, df_test,
        extra_features=None,
        label_extra='[Modelo A — sem IA]',
    )



++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Faixa Salarial [Modelo A — sem IA]
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Treino: 0 | Teste: 4,795 | Classes: []

  [Cross-Validation 5-fold]


TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

## 6. Modelo B - Treino e Teste em 2024 (features base + IA)

In [ ]:
from sklearn.model_selection import train_test_split

results_B = {}

for target, task_type in TARGETS.items():
    print(f'\n{"+"*60}')
    print(f'  {TARGET_LABELS[target]} — [Modelo B — base + IA] (2024 only)')
    print(f'{"+"*60}')

    X_all, y_all, meta_all = prepare_xy(df_test, target,
                                         extra_features=ia_disponivel)
    # Stratify por target E gênero (usa combinação como strato)
    strat = y_all.astype(str) + '_' + meta_all['genero']

    X_tr, X_te, y_tr, y_te, meta_tr, meta_te = train_test_split(
        X_all, y_all, meta_all,
        test_size=0.30, random_state=SEED, stratify=strat,
    )

    print(f'  Treino 2024: {len(X_tr):,} | Teste 2024: {len(X_te):,}')
    print(f'  Features de IA incluídas: {len(ia_disponivel)}')

    print(f'\n  [Cross-Validation 5-fold]')
    models   = get_models(task_type)
    cv_df    = run_cv(X_tr, y_tr, models)
    best_name = cv_df.loc[cv_df['F1_macro_mean'].idxmax(), 'Modelo']
    print(f'\n  🏆 Melhor modelo: {best_name}')

    pre  = build_preprocessor(X_tr)
    pipe = Pipeline([('pre', pre), ('clf', models[best_name])])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)

    print(f'\n  [Classificação — Teste (30% do 2024)]')
    print(classification_report(y_te, y_pred, zero_division=0))

    gender_rows = []
    for g in ['Masculino', 'Feminino']:
        mask = meta_te['genero'] == g
        if mask.sum() == 0: continue
        gender_rows.append({
            'Gênero':   g, 'N': int(mask.sum()),
            'Acurácia': accuracy_score(y_te[mask], y_pred[mask]),
            'F1_macro': f1_score(y_te[mask], y_pred[mask],
                                  average='macro', zero_division=0),
        })
    print(f'\n  [Desempenho por Gênero]')
    print(pd.DataFrame(gender_rows).to_string(index=False))

    results_B[target] = {
        'target': target, 'label': f'{TARGET_LABELS[target]} [B+IA]',
        'best_name': best_name, 'pipe': pipe,
        'cv_df': cv_df, 'X_train': X_tr, 'X_test': X_te,
        'y_test': y_te, 'y_pred': y_pred,
        'gender_rows': gender_rows, 'meta_test': meta_te,
    }


## 7. SHAP - Explicabilidade


In [ ]:
def compute_shap(pipe: Pipeline, X: pd.DataFrame,
                  model_name: str, max_samples: int = 600) -> tuple:
    """
    Extrai SHAP values (amostragem p/ eficiência).
    Retorna: mean_abs_shap por feature, array de nomes de features.
    """
    pre = pipe.named_steps['pre']
    clf = pipe.named_steps['clf']
    X_t = pre.transform(X)

    try:
        feat_names = np.array(pre.get_feature_names_out())
    except Exception:
        feat_names = np.array([f'f{i}' for i in range(X_t.shape[1])])

    n   = min(max_samples, X_t.shape[0])
    idx = np.random.RandomState(SEED).choice(X_t.shape[0], n, replace=False)
    X_s = X_t[idx]

    if 'XGBoost' in model_name or 'Random Forest' in model_name:
        exp = shap.TreeExplainer(clf)
        sv  = exp.shap_values(X_s)
    else:
        exp = shap.LinearExplainer(clf, X_s, feature_dependence='independent')
        sv  = exp.shap_values(X_s)

    # Multiclass → média do |SHAP| entre todas as classes
    if isinstance(sv, list):
        mean_abs = np.mean([np.abs(s) for s in sv], axis=0)
    else:
        mean_abs = np.abs(sv)

    return mean_abs.mean(axis=0), feat_names


def plot_shap_bar(mean_shap: np.ndarray, feat_names: np.ndarray,
                  top_n: int = 15, title: str = 'SHAP',
                  fname: str = 'shap.png',
                  highlight_ia: bool = False):
    """
    Barplot horizontal dos top-N features por |SHAP| médio.
    Se highlight_ia=True, pinta features de IA em laranja.
    """
    top_idx = np.argsort(mean_shap)[::-1][:top_n]
    names   = feat_names[top_idx]
    vals    = mean_shap[top_idx]

    # Limpa prefixos do ColumnTransformer
    clean = lambda n: (n.replace('cat__', '')
                        .replace('num__', '')
                        .replace('_cod', ''))
    names_clean = [clean(n) for n in names]

    if highlight_ia:
        colors = ['#F4A43B' if 'ia_' in n else '#4C8FBF' for n in names_clean]
    else:
        colors = ['#4C8FBF'] * len(names_clean)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(names_clean[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(title, fontweight='bold')
    if highlight_ia:
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color='#F4A43B', label='Feature de IA'),
                            Patch(color='#4C8FBF', label='Feature base')],
                  loc='lower right')
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f'  💾 figures/{fname}')


# ── SHAP para cada target — Modelo A e Modelo B ──────────────────────────────
shap_cache = {}

for target in TARGETS:
    label = TARGET_LABELS[target]
    safe  = target.replace(' ', '_')

    # Modelo A
    try:
        r = results_A[target]
        ms, fn = compute_shap(r['pipe'], r['X_test'], r['best_name'])
        shap_cache[f'A_{target}'] = (ms, fn)
        plot_shap_bar(ms, fn,
                      title=f'SHAP — {label} · Modelo A ({r["best_name"]})',
                      fname=f'shap_A_{safe}.png')
    except Exception as e:
        print(f'⚠️ SHAP Modelo A / {label}: {e}')

    # Modelo B (com IA)
    try:
        r = results_B[target]
        ms, fn = compute_shap(r['pipe'], r['X_test'], r['best_name'])
        shap_cache[f'B_{target}'] = (ms, fn)
        plot_shap_bar(ms, fn,
                      title=f'SHAP — {label} · Modelo B + IA ({r["best_name"]})',
                      fname=f'shap_B_{safe}.png',
                      highlight_ia=True)
    except Exception as e:
        print(f'⚠️ SHAP Modelo B / {label}: {e}')


## 8. Análise exploratória — Adoção de IA por Gênero (2024)

Esta seção é exclusiva para o dataset de 2024, explorando como homens e mulheres se diferenciam no uso de ferramentas de IA generativa.


In [ ]:
df_2024 = df_full[df_full['ano_pesquisa'] == '2024'].copy()

# 8.1 — Uso pessoal de IA
print("=== Uso pessoal de IA por Gênero (%) ===")
if 'ia_uso_pessoal' in df_2024.columns:
    ct = pd.crosstab(df_2024['genero'], df_2024['ia_uso_pessoal'], normalize='index') * 100
    display(ct.round(1))

# 8.2 — Features binárias de IA
ia_bin_cols = [c for c in IA_FEATURES if c not in ('ia_uso_pessoal',)
               and df_2024[c].dropna().isin([0, 1, 0.0, 1.0]).all()]

if ia_bin_cols:
    print("\n=== Taxa de adoção (%) de features de IA por Gênero ===")
    adocao = df_2024.groupby('genero')[ia_bin_cols].mean() * 100
    display(adocao.round(1).T.sort_values('Feminino', ascending=False))

In [ ]:
df_24 = df_test.copy()

ia_labels = {
    'ia_gest_independente':   'Uso independente (gestores)',
    'ia_gest_centralizado':   'Uso centralizado (gestores)',
    'ia_gest_copilot_devs':   'Copilots p/ devs (gestores)',
    'ia_gest_prod_externo':   'Produto externo (gestores)',
    'ia_gest_prod_interno':   'Produto interno (gestores)',
    'ia_gest_frente_negocio': 'Frente do negócio (gestores)',
    'ia_gest_nao_prioridade': 'Não é prioridade (gestores)',
    'ia_gest_nao_sei':        'Não sei opinar (gestores)',
    'ia_bar_casos_uso':       'Falta casos de uso',
    'ia_bar_alucinacao':      'Alucinação',
    'ia_bar_regulacao':       'Regulação',
    'ia_bar_privacidade':     'Privacidade',
    'ia_bar_roi':             'ROI não comprovado',
    'ia_bar_dados':           'Dados não prontos',
    'ia_bar_expertise':       'Falta expertise',
    'ia_bar_alta_direcao':    'Alta direção não vê valor',
    'ia_bar_prop_intelectual':'Prop. intelectual',
    'ia_ind_independente':    'Uso independente (individual)',
    'ia_ind_centralizado':    'Uso centralizado (individual)',
    'ia_ind_copilot_devs':    'Copilots p/ devs (individual)',
    'ia_ind_prod_externo':    'Produto externo (individual)',
    'ia_ind_prod_interno':    'Produto interno (individual)',
    'ia_ind_frente_negocio':  'Frente do negócio (individual)',
    'ia_ind_nao_prioridade':  'Não é prioridade (individual)',
    'ia_ind_nao_sei':         'Não sei opinar (individual)',
    'ia_pes_nao_usa':         'Não usa IA pessoal',
    'ia_pes_gratis':          'Usa gratuito',
    'ia_pes_pago_proprio':    'Paga do bolso',
    'ia_pes_pago_empresa':    'Empresa paga',
    'ia_pes_copilot':         'Usa Copilot',
}

def plot_ia_grupo(df: pd.DataFrame, cols: list, titulo: str, fname: str):
    """
    Gráfico de barras duplas: % de marcação de cada flag de IA por gênero.
    Mostra a diferença entre homens e mulheres em cada opção.
    """
    cols_disp = [c for c in cols if c in df.columns and df[c].notna().sum() > 0]
    if not cols_disp:
        print(f'  ⚠️ Nenhuma coluna disponível para: {titulo}')
        return

    adocao = df.groupby('genero')[cols_disp].mean() * 100
    labels  = [ia_labels.get(c, c) for c in cols_disp]
    delta   = adocao.loc['Masculino'] - adocao.loc['Feminino']
    order   = delta.abs().sort_values(ascending=False).index  # ordena por maior diferença

    fig, ax = plt.subplots(figsize=(11, max(4, len(cols_disp) * 0.55)))
    x = np.arange(len(cols_disp))
    w = 0.35

    bars_m = ax.barh(x + w/2,
                     adocao.loc['Masculino', order].values,
                     w, label='Masculino', color=PALETTE['Masculino'])
    bars_f = ax.barh(x - w/2,
                     adocao.loc['Feminino', order].values,
                     w, label='Feminino',  color=PALETTE['Feminino'])

    ax.set_yticks(x)
    ax.set_yticklabels([ia_labels.get(c, c) for c in order], fontsize=9)
    ax.set_xlabel('% de respondentes que marcaram esta opção')
    ax.set_title(titulo, fontweight='bold')
    ax.legend()
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())
    plt.tight_layout()
    plt.savefig(f'figures/{fname}', bbox_inches='tight')
    plt.show()
    print(f'  💾 figures/{fname}')


# ── 8.1 — 3.f: Uso de IA na empresa (perspectiva do gestor) ─────────────────
plot_ia_grupo(df_24, IA_FEATS_GEST,
              titulo='3.f — Tipo de uso de IA na empresa · Perspectiva do Gestor',
              fname='ia_3f_gestores.png')

# ── 8.2 — 3.g: Barreiras para não usar IA (perspectiva do gestor) ───────────
plot_ia_grupo(df_24, IA_FEATS_BAR,
              titulo='3.g — Barreiras para não usar IA · Perspectiva do Gestor',
              fname='ia_3g_barreiras.png')

# ── 8.3 — 4.l: Uso de IA na empresa (perspectiva individual) ────────────────
plot_ia_grupo(df_24, IA_FEATS_IND,
              titulo='4.l — Tipo de uso de IA na empresa · Perspectiva Individual',
              fname='ia_4l_individual.png')

# ── 8.4 — 4.m: Uso pessoal de ChatGPT / Copilot ────────────────────────────
plot_ia_grupo(df_24, IA_FEATS_PES,
              titulo='4.m — Uso pessoal de ChatGPT / Copilot · por Gênero',
              fname='ia_4m_pessoal.png')


In [ ]:
# ── 8.5 — Comparação 3.f vs 4.l por gênero ──────────────────────────────────
# 3.f e 4.l têm as mesmas opções, respondidas por públicos diferentes.
# Aqui comparamos se gestores e não-gestores concordam, e se isso varia por gênero.

pares = [
    ('ia_gest_independente', 'ia_ind_independente', 'Uso independente'),
    ('ia_gest_centralizado',  'ia_ind_centralizado',  'Uso centralizado'),
    ('ia_gest_copilot_devs',  'ia_ind_copilot_devs',  'Copilots p/ devs'),
    ('ia_gest_prod_externo',  'ia_ind_prod_externo',  'Produto externo'),
    ('ia_gest_prod_interno',  'ia_ind_prod_interno',  'Produto interno'),
    ('ia_gest_frente_negocio','ia_ind_frente_negocio','Frente do negócio'),
    ('ia_gest_nao_prioridade','ia_ind_nao_prioridade','Não é prioridade'),
]

rows = []
for col_gest, col_ind, label in pares:
    if col_gest not in df_24.columns or col_ind not in df_24.columns:
        continue
    for g in ['Masculino', 'Feminino']:
        sub = df_24[df_24['genero'] == g]
        rows.append({
            'Opção':   label,
            'Gênero':  g,
            '3.f (gestor) %':    sub[col_gest].mean() * 100,
            '4.l (individual) %':sub[col_ind].mean()  * 100,
            'Δ (gest - ind)':    (sub[col_gest].mean() - sub[col_ind].mean()) * 100,
        })

df_comp = pd.DataFrame(rows)
print("=== Comparação 3.f (gestor) vs 4.l (individual) por gênero ===")
display(df_comp.round(1))
